In [17]:
import pandas as pd
import tarfile
import xml.etree.ElementTree as ET
from xml.dom.minidom import parse, parseString
from collections import Counter
import numpy as np
import re
import csv
from tqdm import tqdm
import csv
import pickle
import matplotlib.pyplot as plt
import time
import os

Relevant links for information: <br>
(1) https://github.com/OP-TED/eForms-SDK

In [18]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
def tar_extract_and_populate(location_input: str):
    """This function takes in a set of tar files, extracts them in a directory of choice and populates a list of files in the directory"""
    
    tar_data_files = [name for name in os.listdir(location_input)]
    xml_files_list=[]
    year = location_input[-4:]
    
    for tar_data_file in tar_data_files:
        #open the tar.gz file as a tar archive
        print("{}\{}".format(location_input, tar_data_file))
        tar_archive = tarfile.open(name=r"{}\{}".format(location_input, tar_data_file), mode="r:gz")

        #extract the tar archive in a directory of choice
        tar_archive.extractall(r"C:\Users\gbolton\Desktop\Data\{}".format(year))

        for file in tar_archive.getnames():
            if file[-4:] == ".xml":
                xml_files_list.append(str(r"C:\Users\gbolton\Desktop\Data\{}\{}".format(year, file)))

        tar_archive.close()
    
    with open(r"C:\Users\gbolton\Desktop\Data\xml_files_list_{}.pickle".format(year), "wb") as f:
        pickle.dump(xml_files_list, f)

    return xml_files_list

In [3]:
def get_xmlns_value(root):
    """This function takes in an xml file and returns the namespace of the file"""
    root_tag = str(root.tag)
    xmlns_value = root_tag.split("}")[0] + "}"
    return xmlns_value

In [4]:
def query_language_filter(xpath_expressions: dict, language: str):
    """This function takes in a dict of xpath queries (format of the dict is relevant), and changes the language attribute filter"""
    
    keys = list(xpath_expressions)
    
    for key in keys:
        if xpath_expressions[key][1] != "option":
            xpath_expressions[key][0] = xpath_expressions[key][0].replace("//*[@LG='EN']", language)
        else:
            for i, query in enumerate(xpath_expressions[key][3]):
                xpath_expressions[key][3][i] = xpath_expressions[key][3][i].replace("//*[@LG='EN']", language)

    return xpath_expressions

In [ ]:
location_input = r"C:\Users\gbolton\Desktop\Data\raw_data"
xml_files_list = tar_extract_and_populate(location_input)

In [12]:
with open(r"C:\Users\gbolton\Desktop\Data\xml_files_list_data_2020_2021.pickle", "rb") as f:
    xml_files_list_2020_2021 = pickle.load(f)

In [13]:
xml_files_list_2020 = []
xml_files_list_2021 = []

for file in xml_files_list_2020_2021:
    if file[-8:-4] == "2020":
        xml_files_list_2020.append(file)
    if file[-8:-4] == "2021":
        xml_files_list_2021.append(file)

-----
LOAD ALL RELEVANT DATA 
-----

-----INITIALIZE ALL THE DATA AND RUN THE CODE TO RETRIEVE THE DATA FROM THE XML FILES-----

In [29]:
xpath_expressions = {"town": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}TOWN", "text", "single"],
    "postcal_code": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}POSTAL_CODE", "text", "single"]}

In [36]:
xpath_expressions = {"official_name": [".//{}FORM_SECTION//{}CONTRACTING_BODY//{}OFFICIALNAME", "text", "single"],
    "country": [".//{}FORM_SECTION//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}COUNTRY", "attribute", "single"],
    "short_descr": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}SHORT_DESCR//{}P", "text", "single"],
    "object_contract/title": [".//{}FORM_SECTION//{}OBJECT_CONTRACT/{}TITLE//{}P", "text", "single"],
    "object_descr/title": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR/{}TITLE//{}P", "text", "multiple"],
    "object_contract/val_estimated_total": [".//{}FORM_SECTION//{}OBJECT_CONTRACT/{}VAL_ESTIMATED_TOTAL", "text", "multiple"],
    "object_contract/val_estimated_total[@currency]": [".//{}FORM_SECTION//{}OBJECT_CONTRACT/{}VAL_ESTIMATED_TOTAL", "attribute", "single"],
    "object_contract/val_object": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR/{}VAL_OBJECT", "text", "multiple"],
    "object_contract/val_object[@currency]": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}VAL_ESTIMATED_TOTAL", "attribute", "single"],
    "object_contract/val_total": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}VAL_TOTAL", "text", "multiple"],
    "object_contract/val_total[@currency]": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}VAL_TOTAL", "attribute", "single"],
    "object_descr/date_start": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}DATE_START", "text", "multiple"],
    "object_descr/date_end": [".//{}FORM_SECTION//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}DATE_END", "text", "multiple"],
    "award_contract/val_total": [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}VALUES/{}VAL_TOTAL", "text", "multiple"],
    "award_contract/val_total[@currency]": [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}VALUES/{}VAL_TOTAL", "attribute", "single"],
    "award_contract/val_estimated_total": [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}VALUES/{}VAL_ESTIMATED_TOTAL", "text", "single"],
    "award_contract/val_estimated_total[@currency]": [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}VALUES/{}VAL_ESTIMATED_TOTAL", "attribute", "single"],
    "object_descr/duration": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}DURATION", "text", "multiple"],
    "ca_type": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY//{}CA_TYPE", "attribute", "single"],
    "ca_activity": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY//{}CA_ACTIVITY", "attribute", "single"],
    "cpv_code": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT//{}CPV_MAIN//{}CPV_CODE", "attribute", "single"],
    "type_contract": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT//{}TYPE_CONTRACT", "attribute", "single"],
    "duration type": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}DURATION", "attribute", "single"],
    
    "ac_criterion": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR/{}AC/{}AC_QUALITY/{}AC_CRITERION", "text", "multiple"],
    "ac_weighting": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR/{}AC/{}AC_QUALITY//{}AC_WEIGHTING", "text", "multiple"],
    "ac_cost": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}AC//{}AC_COST", "text", "multiple"],
    "ac_price": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}AC//{}AC_PRICE", "text", "multiple"],
    "ac_price_weighting": [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}AC//{}AC_PRICE/{}AC_WEIGHTING", "text", "multiple"],
    
    "ca_type_other": ["placeholder", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}CA_TYPE_OTHER"]],
    "ca_activity_other": ["placeholder", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}CA_ACTIVITY_OTHER"]],
    "renewal": ["placeholder", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}NO_RENEWAL", 
                                                    ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR//{}RENEWAL"]],
    "procedure": ["placeholder", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_OPEN",
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}ACCELERATED_PROC",
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_RESTRICTED",
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_COMPETITIVE_NEGOTIATION",
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_COMPETITIVE_DIALOGUE",
                 ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}PT_AWARD_CONTRACT_WITHOUT_CALL"]],
    "recurrent": ["placeholder", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}COMPLEMENTARY_INFO/{}RECURRENT_PROCUREMENT",
                                                      ".//{}FORM_SECTION//*[@LG='EN']//{}COMPLEMENTARY_INFO/{}NO_RECURRENT_PROCUREMENT"]],
    "joint_procurement_involved": ["placedholder", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY//{}JOINT_PROCUREMENT_INVOLVED"]],
    "central_purchasing": ["placedholder", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY//{}CENTRAL_PURCHASING"]],
    "eu_fund": ["placeholder", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR/{}EU_PROGR_RELATED",
                                                    ".//{}FORM_SECTION//*[@LG='EN']//{}OBJECT_CONTRACT/{}OBJECT_DESCR/{}NO_EU_PROGR_RELATED"]],
    "awarded_contract": ["placeholders", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT",
                                                              ".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}NO_AWARDED_CONTRACT"]],
    "framework or dynamic purchasing": ["placeholders", "option", "single", [".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}FRAMEWORK",
                                                                             ".//{}FORM_SECTION//*[@LG='EN']//{}PROCEDURE//{}DPS"]],
    "date_conclusion_contract": [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}DATE_CONCLUSION_CONTRACT", "text", "single"],
    "date_dispatch_notice": [".//{}FORM_SECTION//*[@LG='EN']//{}COMPLEMENTARY_INFO/{}DATE_DISPATCH_NOTICE", "text", "single"],
    "nb_tenders_received": [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}TENDERS//{}NB_TENDERS_RECEIVED", "text", "single"],
    "nb_tenders_received_sme": [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}TENDERS//{}NB_TENDERS_RECEIVED_SME", "text", "single"],
    "lowest offer": [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}VALUES/{}VAL_RANGE_TOTAL/{}LOW", "text","single"],
    "highest offer": [".//{}FORM_SECTION//*[@LG='EN']//{}AWARD_CONTRACT/{}AWARDED_CONTRACT/{}VALUES/{}VAL_RANGE_TOTAL/{}HIGH", "text", "single"],
    "town": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}TOWN", "text", "single"],
    "postcal_code": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}POSTAL_CODE", "text", "single"],
    "town": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}TOWN", "text", "single"],
    "postcal_code": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}POSTAL_CODE", "text", "single"]
}

keys = list(xpath_expressions)

In [37]:
language = ""
xpath_expressions = query_language_filter(xpath_expressions, language)

In [ ]:
xpath_expressions

In [38]:
len(xml_files_list_2021)

557425

In [40]:
data_dict = {}

for i, file in tqdm(enumerate((xml_files_list_2021))):
  
  procurement_notice_data = {}
  #parse the tree and get the root element
  tree = ET.parse(file)
  root = tree.getroot()
  xmlns_value = get_xmlns_value(root)

  for j, key in enumerate(xpath_expressions.keys()):
    query = xpath_expressions[key][0].format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value)
    query_type = xpath_expressions[key][1]
    quantity = xpath_expressions[key][2]

    if query_type == "text":
      procurement_text = None
      if quantity == "single":
        procurement_notice_data[key] = None
        for k, contract_info in enumerate(root.findall(query)):
          if contract_info.text == None:
            continue
          elif contract_info.text != None and procurement_text != None:
            procurement_text = ". ".join([procurement_text, contract_info.text])
          else:
            procurement_text = contract_info.text
        procurement_notice_data[key] = procurement_text
      else:
        procurement_notice_data[key + ": 0"] = None
        for k, contract_info in enumerate(root.findall(query)):
          if contract_info.text == None:
            procurement_notice_data[key + ": " + str(k)] = procurement_text
          else:
            procurement_text = contract_info.text
            procurement_notice_data[key + ": " + str(k)] = procurement_text
    elif query_type == "attribute":
      procurement_attribute = None
      for k, contract_info in enumerate(root.findall(query)):
        if contract_info.attrib.values() != None:
          procurement_attribute = list(contract_info.attrib.values())[0]
        else:
          continue
      procurement_notice_data[key] = procurement_attribute
    elif query_type == "option":
      procurement_notice_data[key] = None
      for query in xpath_expressions[key][3]:  
        query = query.format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value)
        if len(root.findall(query)) > 0:
          category = query.split("}")[-1]
          procurement_notice_data[key] = category
        else:
          continue
    else:
        continue

  data_dict[i] = procurement_notice_data

557425it [1:10:15, 132.23it/s]


In [35]:
with open("Data/data_dict_2021_new", "wb") as f:
    pickle.dump(data_dict, f)

In [8]:
with open("Data/data_dict_2021.pickle", "rb") as f:
    data_dict_2021 = pickle.load(f)

with open("Data/data_dict_2020.pickle", "rb") as f:
    data_dict_2020 = pickle.load(f)

In [24]:
ac_criterion_cols = []

for i, outer_key in enumerate(data_dict_2020.keys()):
    for inner_key in data_dict_2020[outer_key]:
        if "ac_criterion" in inner_key and "ac_cost" not in inner_key and inner_key not in ac_criterion_cols:
            ac_criterion_cols.append(inner_key)

len(ac_criterion_cols)

1282

In [7]:
#of which columns are there multiple?
repeated_columns = []
numbers = []

for expression in xpath_expressions:
    if xpath_expressions[expression][2] == "multiple":
        repeated_columns.append(expression)

for index in tqdm(data_dict_2020):
    remove_keys = []
    for key in data_dict_2020[index].keys():
        column_base = key.split(":")
        if column_base[0] in repeated_columns:
            if int(column_base[1]) > 30:
                remove_keys.append(key)
            else:
                continue
        else:
            continue

    for key_ in remove_keys:
        data_dict_2020[index].pop(key_)

#transform dict of dicts to a list of dictionaries
data_list = []

for row in data_dict_2020:
    data_list.append(data_dict_2020[row])

df_data = pd.DataFrame.from_records(data_list)

#save the dataframe
df_data.to_pickle("Data/df_data_2020.pickle")

100%|██████████| 643554/643554 [00:24<00:00, 26778.88it/s]


In [ ]:
apparently I have been duplicating the data from various languages in the contract. how to only get the first encounter of the data? 

In [ ]:
THIS IS ALL LOOKING KINDA FUNKY SO LETS CIRCLE BACK TO THE ORIGINAL DATA TO SEE WHAT THAT LOOKS LIKE

In [4]:
with open("Data/xml_files_list_2020.pickle", "rb") as f:
    xml_files_list_2020 = pickle.load(f)

In [ ]:
new_xml_file_list_2020 = []
new_path = r"C:\Users\gbolton\Desktop\Data\data"

for file in xml_files_list_2020:
    temp_list = file.split("/")[3:]
    new_file_path = "{}\{}\{}".format(new_path, temp_list[0], temp_list[1])
    new_xml_file_list_2020.append(new_file_path)

looks like using the [1] at award contract reduces the number of strange cases we find from 371 to 30 in 10000 rows. let's take a look at these cases that remain

In [ ]:
query = ".//{}FORM_SECTION//*"
#print all attributes, values and texts of x elements in data_dict
indices_weird_val_total_2 = []

for i, file in enumerate(tqdm(xml_files_list_2020[:10000])):
    #parse the tree and get the root element
    tree = ET.parse(file)
    root = tree.getroot()
    
    #get the root tag and split it to get the value for the xmlns attribute
    root_tag = str(root.tag)
    xmlns_value = root_tag.split("}")[0] + "}"
    tender_vals_total = []

    for j, contract_info in enumerate(root.findall(query.format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value))):
        tender_vals_total.append(contract_info.text)
    
    for k, value_1 in enumerate(tender_vals_total):
        for l, value_2 in enumerate(tender_vals_total):
            if value_1 == value_2 and k != l:
                indices_weird_val_total_2.append(file)

indices_weird_val_total_2 = list(set(indices_weird_val_total_2))



In [ ]:
data_dict_2020[3]

In [ ]:
    "town": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}TOWN", "text", "single"],
    "postcal_code": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}POSTAL_CODE", "text", "single"],
    "nuts code": [".//{}FORM_SECTION//*[@LG='EN']//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}NUTS", "text", "single"],

In [23]:
query = ".//{}FORM_SECTION//{}CONTRACTING_BODY/{}ADDRESS_CONTRACTING_BODY/{}NUTS"
#print all attributes, values and texts of x elements in data_dict
indices_weird_val_total_2 = []

for i, file in enumerate(tqdm(xml_files_list_2020[:1000])):
    #parse the tree and get the root element
    tree = ET.parse(file)
    root = tree.getroot()
    
    #get the root tag and split it to get the value for the xmlns attribute
    root_tag = str(root.tag)
    xmlns_value = root_tag.split("}")[0] + "}"
    tender_vals_total = []

    for j, contract_info in enumerate(root.findall(query.format(xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value, xmlns_value))):
        print(i, file, contract_info.tag, contract_info.text, contract_info.attrib)

100%|██████████| 1000/1000 [00:00<00:00, 2408.19it/s]


In [ ]:
print(len(indices_weird_val_total), len(indices_weird_val_total_2))

In [ ]:
indices_weird_val_total = list(set(indices_weird_val_total))

In [24]:
with open("Data/Unique tags and frequencies.pickle", "rb") as f:
    unique_tags_and_frequencies = pickle.load(f)

In [ ]:
unique_tags_and_frequencies